# v1 Sparse Retrieval

Dense (Qdrant/Ollama embeddings) + sparse (BM25) retrieval over the seeded doc set, combined via Reciprocal Rank Fusion (RRF).

In [1]:
import sys

sys.path.insert(0, "..")

from rank_bm25 import BM25Okapi

from app.config import settings
from app.services.llm import get_llm_service
from app.services.qdrant.factory import get_qdrant_service

/Users/michaeleko/Documents/Works/ariapay/ariabot/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load corpus from Qdrant

Pull every seeded point (source/heading/text) so BM25 has a corpus and dense search has a baseline to compare against.

In [2]:
from qdrant_client.http.models import FieldCondition, Filter, MatchValue

from app.constants import POINT_TYPE_DOC

service = get_qdrant_service()

points = []
offset = None
while True:
    batch, offset = service.client.scroll(
        service.collection_name,
        scroll_filter=Filter(
            must=[FieldCondition(key="metadata.type", match=MatchValue(value=POINT_TYPE_DOC))]
        ),
        limit=1000,
        offset=offset,
        with_payload=True,
        with_vectors=False,
    )
    points.extend(batch)
    if offset is None:
        break
docs = [
    {
        "id": p.id,
        "source": p.payload["metadata"]["source"],
        "heading": p.payload["metadata"]["heading"],
        "text": p.payload["page_content"],
    }
    for p in points
]
print(f"{len(docs)} docs loaded")
docs[0]

28 docs loaded


{'id': '014ed301-1cb9-8c7b-dad5-42fff6f0e82e',
 'source': 'privacy.md',
 'heading': "8. Children's Privacy",
 'text': "8. Children's Privacy\n\nAriapay is not intended for use by anyone under 18 years of age, and we do not knowingly collect personal data from children."}

## Sparse retrieval (BM25)

Sparse search (BM25) — classic keyword search. Scores docs by term overlap + rarity (rare shared words score higher, common words discounted). Catches exact term matches dense sometimes misses.



In [3]:
def tokenize(text: str) -> list[str]:
    return text.lower().split()


corpus_tokens = [tokenize(f"{d['heading']} {d['text']}") for d in docs]
bm25 = BM25Okapi(corpus_tokens)


def sparse_search(query: str, top_k: int = 10) -> list[tuple[int, float]]:
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return [(i, scores[i]) for i in ranked[:top_k] if scores[i] > 0]

## Dense retrieval (Qdrant + Ollama embeddings)

Dense search — query text → embedding vector (via Ollama, 4096 dims) → Qdrant finds closest doc vectors by cosine similarity. Catches semantic/meaning matches even w/ different wording ("kids" ~ "children").



In [4]:
from app.constants import DOCS_VECTOR_NAME

id_to_idx = {d["id"]: i for i, d in enumerate(docs)}


def dense_search(query: str, top_k: int = 10) -> list[tuple[int, float]]:
    vector = get_llm_service().embed(query)
    hits = service.client.query_points(
        collection_name=service.collection_name,
        query=vector,
        using=DOCS_VECTOR_NAME,
        limit=top_k,
    )
    return [(id_to_idx[h.id], h.score) for h in hits.points if h.id in id_to_idx]

## RRF fusion

`score(d) = sum over rankers of 1 / (k + rank(d))`, rank is 1-indexed. Standard `k=60`.

RRF fusion — run both, get two ranked lists (top 20 each). For each doc, score = sum of 1/(60+rank) across whichever list(s) it appears in. Doc ranked #1 in both lists beats doc ranked #1 in only one. No need to normalize/compare raw scores (BM25 scores and cosine scores aren't on same scale) — RRF only cares about rank position, sidesteps that problem entirely.

Final top-5 = fused ranking, best of both worlds: semantic recall + exact keyword precision.



In [5]:
def rrf_fuse(
    rankings: list[list[tuple[int, float]]], k: int = 60
) -> list[tuple[int, float]]:
    scores: dict[int, float] = {}
    for ranking in rankings:
        for rank, (idx, _) in enumerate(ranking, start=1):
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda kv: kv[1], reverse=True)

In [6]:
def hybrid_search(query: str, top_k: int = 5):
    sparse_hits = sparse_search(query, top_k=20)
    dense_hits = dense_search(query, top_k=20)
    fused = rrf_fuse([dense_hits, sparse_hits])[:top_k]

    print(f"Query: {query!r}\n")
    print("-- dense top --")
    for idx, score in dense_hits[:5]:
        d = docs[idx]
        print(f"  {score:.4f}  {d['source']} / {d['heading']}")

    print("\n-- sparse (BM25) top --")
    for idx, score in sparse_hits[:5]:
        d = docs[idx]
        print(f"  {score:.4f}  {d['source']} / {d['heading']}")

    print("\n-- RRF fused top --")
    for idx, score in fused:
        d = docs[idx]
        print(f"  {score:.4f}  {d['source']} / {d['heading']}")

    return fused

## Try it

In [7]:
_ = hybrid_search("how do you handle children's privacy")

Query: "how do you handle children's privacy"

-- dense top --
  0.5792  privacy.md / 8. Children's Privacy
  0.5322  privacy.md / 10. Contact Us
  0.5179  privacy.md / 9. Changes to This Policy
  0.4905  privacy.md / 6. Your Rights
  0.4903  privacy.md / 7. Cookies and Tracking

-- sparse (BM25) top --
  8.9639  privacy.md / 8. Children's Privacy
  8.2126  privacy.md / 10. Contact Us
  3.4307  terms.md / 8. Data Privacy
  2.7574  privacy.md / 2. How We Use Your Information
  2.6332  terms.md / 1. Acceptance of Terms

-- RRF fused top --
  0.0328  privacy.md / 8. Children's Privacy
  0.0323  privacy.md / 10. Contact Us
  0.0306  privacy.md / 9. Changes to This Policy
  0.0305  privacy.md / 7. Cookies and Tracking
  0.0301  privacy.md / 


In [8]:
_ = hybrid_search("what cookies and tracking technologies are used")

Query: 'what cookies and tracking technologies are used'

-- dense top --
  0.7420  privacy.md / 7. Cookies and Tracking
  0.5193  privacy.md / 10. Contact Us
  0.5162  privacy.md / 1. Information We Collect
  0.5111  privacy.md / 2. How We Use Your Information
  0.4991  privacy.md / 4. Data Security

-- sparse (BM25) top --
  14.6092  privacy.md / 7. Cookies and Tracking
  4.6227  about.md / What we stand for
  2.7800  privacy.md / 4. Data Security
  2.0963  terms.md / 2. Description of Service
  1.9030  terms.md / 8. Data Privacy

-- RRF fused top --
  0.0328  privacy.md / 7. Cookies and Tracking
  0.0313  privacy.md / 4. Data Security
  0.0298  privacy.md / 1. Information We Collect
  0.0297  privacy.md / 2. How We Use Your Information
  0.0296  about.md / What we stand for


In [9]:
_ = hybrid_search("data privacy and personal information")

Query: 'data privacy and personal information'

-- dense top --
  0.5600  privacy.md / 10. Contact Us
  0.5443  privacy.md / 6. Your Rights
  0.5423  privacy.md / 3. Data Sharing and Disclosure
  0.5189  privacy.md / 5. Data Retention
  0.5113  privacy.md / 

-- sparse (BM25) top --
  7.7372  terms.md / 8. Data Privacy
  6.4561  privacy.md / 8. Children's Privacy
  6.0592  privacy.md / 3. Data Sharing and Disclosure
  4.9481  privacy.md / 5. Data Retention
  4.2863  privacy.md / 1. Information We Collect

-- RRF fused top --
  0.0317  privacy.md / 3. Data Sharing and Disclosure
  0.0313  terms.md / 8. Data Privacy
  0.0312  privacy.md / 5. Data Retention
  0.0311  privacy.md / 6. Your Rights
  0.0306  privacy.md / 8. Children's Privacy


## Generation (pre-rerank, on RRF-fused context)

Generate an answer per query using the top-5 RRF-fused contexts, via the pluggable LLM service (`get_llm_service().chat`). Baseline generation quality before reranking is added.

In [10]:
def contexts_for(idxs: list[int], top_k: int = 5) -> list[str]:
    return [f"{docs[i]['heading']}\n\n{docs[i]['text']}" for i in idxs[:top_k]]


def generate_answer(query: str, contexts: list[str]) -> str:
    context_block = "\n\n".join(contexts)
    prompt = (
        "Answer the question using only the context below. Be concise.\n\n"
        f"Context:\n{context_block}\n\n"
        f"Question: {query}\n\nAnswer:"
    )
    messages = [{"role": "user", "content": prompt}]
    return get_llm_service().chat(messages)


def generate_from_fused(query: str, top_k: int = 5) -> str:
    sparse_hits = sparse_search(query, top_k=20)
    dense_hits = dense_search(query, top_k=20)
    fused = rrf_fuse([dense_hits, sparse_hits])[:top_k]
    contexts = contexts_for([idx for idx, _ in fused], top_k=top_k)

    answer = generate_answer(query, contexts)
    print(f"Query: {query!r}\n\nAnswer: {answer}")
    return answer

In [11]:
_ = generate_from_fused("how do you handle children's privacy")

Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.


Query: "how do you handle children's privacy"

Answer: Ariapay is not intended for use by anyone under 18 years of age, and we do not knowingly collect personal data from children.


In [12]:
_ = generate_from_fused("what cookies and tracking technologies are used")

Query: 'what cookies and tracking technologies are used'

Answer: Cookies and similar technologies are used to keep you signed in, remember your preferences, and understand how our site is used.


In [13]:
_ = generate_from_fused("data privacy and personal information")

Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.


Query: 'data privacy and personal information'

Answer: Ariapay collects and processes personal data in accordance with its Privacy Policy and applicable data protection laws. It uses industry-standard encryption and is PCI DSS Level 1 certified.


## Late-interaction reranking (ColBERT, via Qdrant native multivector)

Rerank the RRF-fused candidates with a ColBERT-style late-interaction model. Each doc/query gets one vector *per token* instead of one pooled vector; relevance is MaxSim — for every query token, take its best match among the doc's tokens, then sum. Captures finer-grained term interactions than a single dense vector or BM25 alone.

Scored using Qdrant's own local multivector engine (an in-memory collection with `MultiVectorConfig` / `MAX_SIM` comparator) — the same scoring path Qdrant uses server-side for native late-interaction search — rather than hand-rolled MaxSim math. Only the fused top-N candidates get ColBERT vectors computed, not the whole corpus, so it stays cheap.

In [14]:
from fastembed import LateInteractionTextEmbedding
from qdrant_client import QdrantClient, models as qm

colbert = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")
COLBERT_DIM = 128


def rerank_late_interaction(
    query: str, candidate_idxs: list[int], top_k: int = 5
) -> list[tuple[int, float]]:
    if not candidate_idxs:
        return []

    query_vec = next(colbert.query_embed(query)).tolist()
    doc_texts = [f"{docs[i]['heading']}\n\n{docs[i]['text']}" for i in candidate_idxs]
    doc_vecs = [v.tolist() for v in colbert.embed(doc_texts)]

    rr = QdrantClient(":memory:")
    rr.create_collection(
        "rerank",
        vectors_config=qm.VectorParams(
            size=COLBERT_DIM,
            distance=qm.Distance.COSINE,
            multivector_config=qm.MultiVectorConfig(
                comparator=qm.MultiVectorComparator.MAX_SIM
            ),
        ),
    )
    rr.upsert(
        "rerank",
        points=[qm.PointStruct(id=i, vector=vec) for i, vec in enumerate(doc_vecs)],
    )
    hits = rr.query_points("rerank", query=query_vec, limit=top_k).points
    return [(candidate_idxs[h.id], h.score) for h in hits]


def hybrid_search_reranked(query: str, top_k: int = 5, candidate_pool: int = 20):
    sparse_hits = sparse_search(query, top_k=candidate_pool)
    dense_hits = dense_search(query, top_k=candidate_pool)
    fused = rrf_fuse([dense_hits, sparse_hits])
    fused_idxs = [idx for idx, _ in fused[:candidate_pool]]

    reranked = rerank_late_interaction(query, fused_idxs, top_k=top_k)

    print(f"Query: {query!r}\n")
    print("-- RRF fused top (pre-rerank) --")
    for idx, score in fused[:top_k]:
        d = docs[idx]
        print(f"  {score:.4f}  {d['source']} / {d['heading']}")

    print("\n-- reranked (ColBERT late-interaction) top --")
    for idx, score in reranked:
        d = docs[idx]
        print(f"  {score:.4f}  {d['source']} / {d['heading']}")

    return reranked

## Try it (reranked) — same queries, compare against the fused-only results above

In [15]:
_ = hybrid_search_reranked("how do you handle children's privacy")

Query: "how do you handle children's privacy"

-- RRF fused top (pre-rerank) --
  0.0328  privacy.md / 8. Children's Privacy
  0.0323  privacy.md / 10. Contact Us
  0.0308  privacy.md / 7. Cookies and Tracking
  0.0306  privacy.md / 9. Changes to This Policy
  0.0301  privacy.md / 

-- reranked (ColBERT late-interaction) top --
  20.3138  privacy.md / 8. Children's Privacy
  15.2751  privacy.md / 10. Contact Us
  13.5740  terms.md / 8. Data Privacy
  11.6121  terms.md / 1. Acceptance of Terms
  11.0579  terms.md / 3. Account Registration


In [16]:
_ = hybrid_search_reranked("what cookies and tracking technologies are used")

Query: 'what cookies and tracking technologies are used'

-- RRF fused top (pre-rerank) --
  0.0328  privacy.md / 7. Cookies and Tracking
  0.0313  privacy.md / 4. Data Security
  0.0298  privacy.md / 1. Information We Collect
  0.0297  privacy.md / 2. How We Use Your Information
  0.0296  about.md / What we stand for

-- reranked (ColBERT late-interaction) top --
  26.2826  privacy.md / 7. Cookies and Tracking
  11.7143  terms.md / 2. Description of Service
  9.6387  privacy.md / 4. Data Security
  8.9072  privacy.md / 1. Information We Collect
  8.9049  terms.md / 8. Data Privacy


In [17]:
_ = hybrid_search_reranked("privasi data dan informasi pribadi")

Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.


Query: 'privasi data dan informasi pribadi'

-- RRF fused top (pre-rerank) --
  0.0318  privacy.md / 3. Data Sharing and Disclosure
  0.0318  privacy.md / 5. Data Retention
  0.0313  privacy.md / 4. Data Security
  0.0309  terms.md / 8. Data Privacy
  0.0301  privacy.md / 8. Children's Privacy

-- reranked (ColBERT late-interaction) top --
  11.6229  terms.md / 8. Data Privacy
  9.4287  privacy.md / 4. Data Security
  8.8198  about.md / What we stand for
  8.1720  privacy.md / 5. Data Retention
  8.1374  terms.md / 3. Account Registration


## Generation (post-rerank, on ColBERT-reranked context)

Same generation, now fed the top-5 reranked contexts instead of raw RRF fusion. Compare answers against the pre-rerank generations above.

In [18]:
def generate_from_reranked(
    query: str, top_k: int = 5, candidate_pool: int = 20
) -> str:
    sparse_hits = sparse_search(query, top_k=candidate_pool)
    dense_hits = dense_search(query, top_k=candidate_pool)
    fused = rrf_fuse([dense_hits, sparse_hits])
    fused_idxs = [idx for idx, _ in fused[:candidate_pool]]

    reranked = rerank_late_interaction(query, fused_idxs, top_k=top_k)
    contexts = contexts_for([idx for idx, _ in reranked], top_k=top_k)

    answer = generate_answer(query, contexts)
    print(f"Query: {query!r}\n\nAnswer: {answer}")
    return answer

In [19]:
_ = generate_from_reranked("how do you handle children's privacy")

Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.


Query: "how do you handle children's privacy"

Answer: Ariapay is not intended for use by anyone under 18 years of age, and we do not knowingly collect personal data from children.


In [20]:
_ = generate_from_reranked("what cookies and tracking technologies are used")

Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.


Query: 'what cookies and tracking technologies are used'

Answer: Cookies and similar technologies to keep you signed in, remember your preferences, and understand how our site is used.


In [21]:
_ = generate_from_reranked("data privacy and personal information")

Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.


Query: 'data privacy and personal information'

Answer: Ariapay collects and processes personal data in accordance with its Privacy Policy and applicable data protection laws, and uses industry-standard encryption.


## Evaluation (RAGAS)

Full RAG eval: generate an answer per query using retrieved context, then score with RAGAS's LLM-judged metrics.

- **Context Precision** — are the relevant chunks ranked near the top of what was retrieved
- **Context Recall** — did retrieval surface everything needed to support the reference answer
- **Faithfulness** — is the generated answer grounded in the retrieved context (hallucination check)
- **Answer Relevancy** — does the answer actually address the question
- **Answer Correctness** — does the answer match the reference answer (factual + semantic)
- **Semantic Similarity** — embedding similarity between generated answer and reference answer

Judge LLM: local `llama3.1:8b` via Ollama. Judge/similarity embeddings: local `qwen3-embedding:8b` via Ollama. Both wired through `langchain_ollama` + ragas's `LangchainLLMWrapper`/`LangchainEmbeddingsWrapper`.

Run once per retrieval method (dense, sparse, RRF-fused, reranked) over the same hand-labeled query set, so the metrics are directly comparable across methods.

In [22]:
EVAL_SET = [
    {
        "query": "how do you handle children's privacy",
        "reference": "Ariapay is not intended for anyone under 18 and does not knowingly collect personal data from children.",
    },
    {
        "query": "what cookies and tracking technologies are used",
        "reference": "Ariapay's website uses cookies and similar technologies to keep users signed in, remember preferences, and understand site usage. Users can control cookies via browser settings.",
    },
    {
        "query": "data privacy and personal information",
        "reference": "Ariapay collects and processes personal data per its Privacy Policy and applicable laws like GDPR, using industry-standard encryption and PCI DSS Level 1 certification. Personal data is not sold to third parties; it's shared only with payment networks, regulators, and necessary service providers.",
    },
]

In [26]:
import instructor
import litellm
from ragas.llms import LiteLLMStructuredLLM
from ragas.embeddings import LiteLLMEmbeddings
from ragas.metrics.collections import (
    ContextPrecisionWithoutReference,
    ContextRecall,
    Faithfulness,
    AnswerRelevancy,
    AnswerCorrectness,
    SemanticSimilarity,
)

from app.constants import DEEPINFRA_OPENAI_BASE, LLMProvider, OPENAI_MODEL_PREFIX

_is_ollama = settings.LLM_PROVIDER == LLMProvider.OLLAMA

_instructor_mode = instructor.Mode.TOOLS if _is_ollama else instructor.Mode.JSON
judge_client = instructor.from_litellm(litellm.acompletion, mode=_instructor_mode)
judge_llm = LiteLLMStructuredLLM(
    client=judge_client,
    model=settings.CHAT_MODEL,
    provider=settings.LLM_PROVIDER,
    api_base=settings.OLLAMA_URL if _is_ollama else None,
    api_key=None if _is_ollama else settings.DEEPINFRA_API_TOKEN,
)

judge_embeddings = LiteLLMEmbeddings(
    model=(
        settings.EMBED_MODEL
        if _is_ollama
        else f"{OPENAI_MODEL_PREFIX}{settings.DEEPINFRA_EMBED_MODEL}"
    ),
    api_base=settings.OLLAMA_URL if _is_ollama else DEEPINFRA_OPENAI_BASE,
    api_key=None if _is_ollama else settings.DEEPINFRA_API_TOKEN,
)

RAGAS_METRICS = [
    ContextPrecisionWithoutReference(llm=judge_llm),
    ContextRecall(llm=judge_llm),
    Faithfulness(llm=judge_llm),
    AnswerRelevancy(llm=judge_llm, embeddings=judge_embeddings),
    AnswerCorrectness(llm=judge_llm, embeddings=judge_embeddings),
    SemanticSimilarity(embeddings=judge_embeddings),
]

In [27]:
import inspect

from ragas.metrics.result import MetricResult


async def score_sample(
    metric, user_input: str, response: str, retrieved_contexts: list[str], reference: str
) -> MetricResult:
    kwargs = {}
    params = inspect.signature(metric.ascore).parameters
    if "user_input" in params:
        kwargs["user_input"] = user_input
    if "response" in params:
        kwargs["response"] = response
    if "retrieved_contexts" in params:
        kwargs["retrieved_contexts"] = retrieved_contexts
    if "reference" in params:
        kwargs["reference"] = reference
    return await metric.ascore(**kwargs)


def build_samples_for_method(
    method_name: str, top_k: int = 5
) -> list[dict]:
    samples = []
    for item in EVAL_SET:
        query, reference = item["query"], item["reference"]

        if method_name == "dense":
            hits = dense_search(query, top_k=top_k)
            idxs = [i for i, _ in hits]
        elif method_name == "sparse":
            hits = sparse_search(query, top_k=top_k)
            idxs = [i for i, _ in hits]
        elif method_name == "rrf":
            sparse_hits = sparse_search(query, top_k=20)
            dense_hits = dense_search(query, top_k=20)
            fused = rrf_fuse([dense_hits, sparse_hits])
            idxs = [i for i, _ in fused[:top_k]]
        elif method_name == "reranked":
            sparse_hits = sparse_search(query, top_k=20)
            dense_hits = dense_search(query, top_k=20)
            fused = rrf_fuse([dense_hits, sparse_hits])
            fused_idxs = [i for i, _ in fused[:20]]
            reranked = rerank_late_interaction(query, fused_idxs, top_k=top_k)
            idxs = [i for i, _ in reranked]
        else:
            raise ValueError(f"unknown method {method_name!r}")

        contexts = contexts_for(idxs, top_k=top_k)
        answer = generate_answer(query, contexts)

        samples.append(
            {
                "user_input": query,
                "retrieved_contexts": contexts,
                "response": answer,
                "reference": reference,
            }
        )
    return samples


async def eval_method(method_name: str, top_k: int = 5) -> list[dict]:
    samples = build_samples_for_method(method_name, top_k=top_k)
    rows = []
    for sample in samples:
        row = {}
        for metric in RAGAS_METRICS:
            result = await score_sample(metric, **sample)
            row[metric.name] = result.value
        rows.append(row)
    return rows

### Run eval across all 4 methods

Each call generates 3 answers (one per eval query) + runs 6 LLM-judged metrics per sample — this makes real Ollama calls and will take a few minutes.

In [28]:
METHODS = ["dense", "sparse", "rrf", "reranked"]

eval_results = {}
for method in METHODS:
    print(f"Evaluating {method}...")
    eval_results[method] = await eval_method(method)
    print(eval_results[method])

Evaluating dense...


Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact

[{'context_precision_without_reference': 0.9999999999, 'context_recall': 1.0, 'faithfulness': 1.0, 'answer_relevancy': 0.4610662771907539, 'answer_correctness': 0.9974853391571535, 'semantic_similarity': 0.9901401463513342}, {'context_precision_without_reference': 0.7555555555303703, 'context_recall': 1.0, 'faithfulness': 0.3333333333333333, 'answer_relevancy': 0.788237101339437, 'answer_correctness': 0.5243722918836535, 'semantic_similarity': 0.5988366741680575}, {'context_precision_without_reference': 0.249999999975, 'context_recall': 1.0, 'faithfulness': 0.6666666666666666, 'answer_relevancy': 0.5697833906408833, 'answer_correctness': 0.28566806743965845, 'semantic_similarity': 0.7411353507823424}]
Evaluating sparse...


Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact

[{'context_precision_without_reference': 0.9999999999, 'context_recall': 1.0, 'faithfulness': 1.0, 'answer_relevancy': 0.47224104649312365, 'answer_correctness': 0.7475082966565565, 'semantic_similarity': 0.990284901671256}, {'context_precision_without_reference': 0.6999999999766667, 'context_recall': 1.0, 'faithfulness': 0.3333333333333333, 'answer_relevancy': 0.7781709889535485, 'answer_correctness': 0.5247828689489371, 'semantic_similarity': 0.5991314757957484}, {'context_precision_without_reference': 0.9999999999, 'context_recall': 1.0, 'faithfulness': 1.0, 'answer_relevancy': 0.5279741236121837, 'answer_correctness': 0.6494409636014222, 'semantic_similarity': 0.9610638875353541}]
Evaluating rrf...


Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact

[{'context_precision_without_reference': 0.9999999999, 'context_recall': 1.0, 'faithfulness': 1.0, 'answer_relevancy': 0.43840004446862807, 'answer_correctness': 0.9975494069700881, 'semantic_similarity': 0.9901976278803524}, {'context_precision_without_reference': 0.8666666666377778, 'context_recall': 1.0, 'faithfulness': 0.5, 'answer_relevancy': 0.7240746773155173, 'answer_correctness': 0.8085845381008743, 'semantic_similarity': 0.6619293598380894}, {'context_precision_without_reference': 0.49999999995, 'context_recall': 1.0, 'faithfulness': 1.0, 'answer_relevancy': 0.5840007535289763, 'answer_correctness': 0.7461898659248423, 'semantic_similarity': 0.8428756109892757}]
Evaluating reranked...


Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact support: https://langfuse.com/support.
Unexpected error occurred. Please check your request and contact

[{'context_precision_without_reference': 0.99999999998, 'context_recall': 1.0, 'faithfulness': 1.0, 'answer_relevancy': 0.48057782444319486, 'answer_correctness': 0.8475544912797939, 'semantic_similarity': 0.9901671324651178}, {'context_precision_without_reference': 0.8333333332916666, 'context_recall': 1.0, 'faithfulness': 0.0, 'answer_relevancy': 0.7069272581876594, 'answer_correctness': 0.6614831268181569, 'semantic_similarity': 0.6472568259864817}, {'context_precision_without_reference': 0.99999999995, 'context_recall': 1.0, 'faithfulness': 1.0, 'answer_relevancy': 0.5399840937521688, 'answer_correctness': 0.48338135236447105, 'semantic_similarity': 0.8426315665548233}]


### Comparison table

In [29]:
import pandas as pd

summary = pd.DataFrame(
    {
        method: pd.DataFrame(rows).mean(numeric_only=True)
        for method, rows in eval_results.items()
    }
).T
summary

,context_precision_without_reference,context_recall,faithfulness,answer_relevancy,answer_correctness,semantic_similarity
dense,0.668519,1.0,0.666667,0.606362,0.602509,0.776704
sparse,0.900000,1.0,0.777778,0.592795,0.640577,0.850160
rrf,0.788889,1.0,0.833333,0.582158,0.850775,0.831668
reranked,0.944444,1.0,0.666667,0.575830,0.664140,0.826685
